In [0]:

WITH FilteredAndCleaned AS (
    SELECT 
        IncidentId,
        COALESCE(DispatchTimestamp, LogCreatedTimeStamp) AS NormalizedTimestamp,
        ROW_NUMBER() OVER (
            PARTITION BY IncidentID 
            ORDER BY DispatchTimestamp ASC
        ) as DuplicateRank
    FROM CAD_Dispatches    
    WHERE RegionName = 'Mississauga'
      -- FIXED: Using COALESCE here ensures rows with NULL DispatchTimestamps aren't dropped!
      AND COALESCE(DispatchTimestamp, LogCreatedTimeStamp) >= '2025-01-01'
      AND COALESCE(DispatchTimestamp, LogCreatedTimeStamp) < '2026-01-01'
),
MergedAnalyticsDataset AS (
    SELECT 
        c.IncidentID,
        DATE_TRUNC(month, c.NormalizedTimestamp) AS DispatchMonth,
        ISNULL(NULLIF(TRIM(n.OffenseCategory), ''), 'UNCLASSIFIED') AS CleanedCategory
    FROM FilteredAndCleaned c
    JOIN Niche_Incidents n 
        ON c.IncidentID = n.IncidentID
    WHERE c.DuplicateRank = 1
),
MonthlyCategory AS (
    SELECT 
        DispatchMonth,
        CleanedCategory,
        COUNT(IncidentID) AS TotalMonthlyIncidents,
        DENSE_RANK() OVER (
            PARTITION BY DispatchMonth 
            ORDER BY COUNT(IncidentID) DESC
        ) AS VolumetricRank
    FROM MergedAnalyticsDataset
    GROUP BY DispatchMonth, CleanedCategory
)
SELECT 
    DispatchMonth, 
    CleanedCategory, 
    TotalMonthlyIncidents, 
    VolumetricRank
FROM MonthlyCategory
WHERE VolumetricRank <= 3
ORDER BY DispatchMonth ASC, VolumetricRank ASC;














In [0]:
LeetCode Problem: Top Volumetric Crime CategoriesThe Analytics Bureau at Peel Regional Police needs to track and rank monthly crime trends to optimize patrol resource allocation. However, operational systems suffer from real-time data entry gaps and radio transmission glitches.Write a solution to identify the top 3 crime categories in the Mississauga region for each month of 2025 based on total dispatch volume.Database SchemaTable: CAD_Dispatches+---------------------+-----------+

| Column Name         | Type      |
+---------------------+-----------+

| IncidentID          | int       |
| DispatchTimestamp   | datetime  |
| LogCreatedTimestamp | datetime  |
| RegionName          | varchar   |
+---------------------+-----------+
There is no primary key for this table; it may contain duplicate rows.
Each row logs a radio transmission regarding an incident. Due to hardware glitches, 
the same transmission can be logged multiple times with identical timestamps.
LogCreatedTimestamp is never NULL, but DispatchTimestamp can occasionally be NULL 
due to system dropouts.
Table: Niche_Incidents+-----------------+-----------+

| Column Name     | Type      |
+-----------------+-----------+

| IncidentID      | int       |
| OffenseCategory | varchar   |
+-----------------+-----------+
IncidentID is the primary key for this table.
Each row represents a finalized records management profile. 
Due to manual field omissions by officers, OffenseCategory may contain empty strings, 
blank spaces ('   '), or NULL values.
Problem Requirements & Business RulesDeduplication: You must eliminate duplicate radio transmissions. Evaluate only the earliest (first) recorded transmission for any given IncidentID based on chronological order.Missing Timestamps: If DispatchTimestamp is missing (NULL), gracefully fall back to the LogCreatedTimestamp to determine the incident's true date and time.Data Cleansing: Standardize corrupted or blank text entries. If OffenseCategory is an empty string, contains only spaces, or is NULL, it must be classified as 'UNCLASSIFIED'.Volumetric Ranking: For each month of 2025 in the Mississauga region, rank the offense categories from highest to lowest volume. Return only the top 3 categories per month.Tie-Breaker Rule: If multiple crime categories share the exact same incident count in a given month, they should receive the same rank (no rank numbers should be skipped).Example ScenarioInputCAD_Dispatches table:+------------+---------------------+---------------------+-------------+

| IncidentID | DispatchTimestamp   | LogCreatedTimestamp | RegionName  |
+------------+---------------------+---------------------+-------------+

| 101        | 2025-01-15 08:30:00 | 2025-01-15 08:31:00 | Mississauga |
| 102        | 2025-01-16 09:15:00 | 2025-01-16 09:16:00 | Mississauga |
| 103        | 2025-01-17 14:20:00 | 2025-01-17 14:22:00 | Mississauga |
| 104        | 2025-01-18 23:45:00 | 2025-01-18 23:46:00 | Mississauga |
| 105        | 2025-01-20 11:00:00 | 2025-01-20 11:01:00 | Mississauga |
| 105        | 2025-01-20 11:00:00 | 2025-01-20 11:01:00 | Mississauga | <-- Glitch Duplicate
| 201        | 2025-02-02 04:12:00 | 2025-02-02 04:15:00 | Mississauga |
| 202        | 2025-02-10 18:22:00 | 2025-02-10 18:25:00 | Mississauga |
| 203        | NULL                | 2025-02-14 13:00:00 | Mississauga | <-- Missing Timestamp
| 901        | 2024-12-31 23:59:00 | 2024-12-31 23:59:00 | Mississauga | <-- Wrong Year
| 902        | 2025-06-15 12:00:00 | 2025-06-15 12:01:00 | Brampton    | <-- Wrong Region
+------------+---------------------+---------------------+-------------+
Niche_Incidents table:+------------+-----------------+

| IncidentID | OffenseCategory |
+------------+-----------------+

| 101        | VEHICLE_THEFT   |
| 102        | ASSAULT         |
| 103        | VEHICLE_THEFT   |
| 104        | PROPERTY_DAMAGE |
| 105        | ASSAULT         |
| 201        | VEHICLE_THEFT   |
| 202        | BREAK_AND_ENTER |
| 203        |    		   | <-- Corrupted Blank Fragment
| 901        | FRAUD           |
| 902        | ASSAULT         |
+------------+-----------------+
Output+---------------+-----------------+-----------------------+----------------+

| DispatchMonth | CleanedCategory | TotalMonthlyIncidents | VolumetricRank |
+---------------+-----------------+-----------------------+----------------+

| 2025-01-01    | VEHICLE_THEFT   | 2                     | 1              |
| 2025-01-01    | ASSAULT         | 2                     | 1              |
| 2025-01-01    | PROPERTY_DAMAGE | 1                     | 2              |
| 2025-02-01    | BREAK_AND_ENTER | 1                     | 1              |
| 2025-02-01    | UNCLASSIFIED    | 1                     | 1              |
| 2025-02-01    | VEHICLE_THEFT   | 1                     | 1              |
+---------------+-----------------+-----------------------+----------------+
ExplanationFor January 2025:Incident 105 was logged twice due to a radio glitch, but your logic successfully deduplicated it down to 1 distinct count.Both VEHICLE_THEFT (Incidents 101, 103) and ASSAULT (Incident 102, and the deduplicated 105) tied with a total count of 2 incidents. Therefore, they both received a VolumetricRank of 1.PROPERTY_DAMAGE (Incident 104) has a count of 1 and directly follows as rank 2.For February 2025:Incident 203 had a missing timestamp, but fell back safely to its log creation date (2025-02-14), placing it correctly inside February. Its offense category was a blank string, which was securely re-mapped to 'UNCLASSIFIED'.BREAK_AND_ENTER, VEHICLE_THEFT, and 'UNCLASSIFIED' all tied with 1 incident each, grouping them as shared rank 1 leaders for the month.Exclusions:Incident 901 was skipped entirely because it occurred in 2024.Incident 902 was skipped because its boundary tracking belonged to Brampton instead of Mississauga.

In [0]:
WITH FilteredAndCleaned AS(
    
)